In [1]:
%pip install -q pandas beautifulsoup4 chromadb sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


In [2]:
import pandas as pd
import re
from html import unescape
from bs4 import BeautifulSoup
import chromadb
from sentence_transformers import SentenceTransformer

In [3]:
csv_url = "https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset/resolve/main/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"

raw_df = pd.read_csv(csv_url)

print("Shape:", raw_df.shape)

raw_df.head()


Shape: (26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [4]:
faq_df = raw_df[["instruction", "response"]].copy()

faq_df = faq_df.rename(columns={
    "instruction": "question",
    "response": "answer"
})

faq_df.head()

,question,answer
0,question about cancelling order {{Order Number}},I've understood you have a question regarding ...
1,i have a question about cancelling oorder {{Or...,I've been informed that you have a question ab...
2,i need help cancelling puchase {{Order Number}},I can sense that you're seeking assistance wit...
3,I need to cancel purchase {{Order Number}},I understood that you need assistance with can...
4,"I cannot afford this order, cancel purchase {{...",I'm sensitive to the fact that you're facing f...


In [5]:
def clean_text(text):
    text = str(text)
    text = unescape(text)

    if re.search(r"<[a-zA-Z/][^>]*>", text):
        text = BeautifulSoup(text, "html.parser").get_text(" ")

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [6]:
faq_df = faq_df.dropna(subset=["question", "answer"]).copy()

for col in ["question", "answer"]:
    faq_df[col] = faq_df[col].apply(clean_text)

faq_df = faq_df[
    (faq_df["question"] != "") &
    (faq_df["answer"] != "")
].copy()

faq_df = faq_df.drop_duplicates(
    subset=["question", "answer"]
).reset_index(drop=True)

faq_df.head()


,question,answer
0,question about cancelling order {{Order Number}},I've understood you have a question regarding ...
1,i have a question about cancelling oorder {{Or...,I've been informed that you have a question ab...
2,i need help cancelling puchase {{Order Number}},I can sense that you're seeking assistance wit...
3,I need to cancel purchase {{Order Number}},I understood that you need assistance with can...
4,"I cannot afford this order, cancel purchase {{...",I'm sensitive to the fact that you're facing f...


In [7]:
faq_df["document"] = (
    "Question: " + faq_df["question"] +
    "\nAnswer: " + faq_df["answer"]
)

documents = faq_df["document"].tolist()

print("Total documents:", len(documents))
print(documents[0])


Total documents: 26872
Question: question about cancelling order {{Order Number}}
Answer: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.


In [8]:
def split_into_chunks(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))

        if end >= len(words):
            break

        start = end - overlap

    return chunks

In [9]:
chunk_records = []

for index, document in enumerate(documents):
    text_chunks = split_into_chunks(document)

    for chunk_index, chunk in enumerate(text_chunks):
        chunk_records.append({
            "id": f"doc_{index}_chunk_{chunk_index}",
            "text": chunk,
            "source_id": index
        })

print("Total chunks:", len(chunk_records))
display(pd.DataFrame(chunk_records).head())

Total chunks: 44132


,id,text,source_id
0,doc_0_chunk_0,Question: question about cancelling order {{Or...,0
1,doc_1_chunk_0,Question: i have a question about cancelling o...,1
2,doc_2_chunk_0,Question: i need help cancelling puchase {{Ord...,2
3,doc_2_chunk_1,'{{Online Order Interaction}}' section to find...,2
4,doc_2_chunk_2,team is available during {{Customer Support Ho...,2


In [11]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="bitext_faq_practice",
    metadata={"hnsw:space": "cosine"}
)

chunk_texts = [item["text"] for item in chunk_records]
chunk_ids = [item["id"] for item in chunk_records]

chunk_metadatas = [
    {"source_id": item["source_id"]}
    for item in chunk_records
]

batch_size = 256

for start in range(0, len(chunk_texts), batch_size):
    batch_texts = chunk_texts[start:start + batch_size]
    batch_ids = chunk_ids[start:start + batch_size]
    batch_metadata = chunk_metadatas[start:start + batch_size]

    embeddings = model.encode(
        batch_texts,
        normalize_embeddings=True
    ).tolist()

    collection.upsert(
        ids=batch_ids,
        documents=batch_texts,
        metadatas=batch_metadata,
        embeddings=embeddings
    )

print("Stored chunks:", collection.count())


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Stored chunks: 44132


In [12]:
query = "How do I cancel an order?"

query_embedding = model.encode(
    [query],
    normalize_embeddings=True
).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3,
    include=["documents", "distances", "metadatas"]
)

print("Query:", query)
print("\nTop 3 matches:\n")

for rank, (document, distance) in enumerate(
    zip(results["documents"][0], results["distances"][0]),
    start=1
):
    print(f"Rank: {rank}")
    print(f"Cosine distance: {distance:.4f}")
    print(document)
    print("-" * 60)


Query: How do I cancel an order?

Top 3 matches:

Rank: 1
Cosine distance: 0.1467
Orders" section. 3. Locate your order: Look for the order labeled with the order number {{Order Number}} and click on it. 4. Initiate cancellation: Within the order details page, you should find an option to "Cancel Order." Please select this option. 5. Confirm cancellation: The system may prompt you to confirm the cancellation. Make sure to review the details and confirm your request. If you encounter any difficulties or have further questions, our dedicated customer support team is available to assist you during {{Customer Support Hours}}. You can reach us at {{Customer Support Phone Number}} or through the Live Chat
------------------------------------------------------------
Rank: 2
Cosine distance: 0.1481
find the section that lists all your orders, typically labeled as "Order History" or "My Orders". 3. Locate the relevant purchase: Scan through the list and identify the purchase associated with the

In [ ]:
df = pd.read_csv("test/test_customer_support.csv")

for col in ["query"]:
    query = df[col]
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3,
        include=["documents", "distances", "metadatas"]
    )

    print("Query:", query)
    print("\nTop 3 matches:\n")

    for rank, (document, distance) in enumerate(
        zip(results["documents"][0], results["distances"][0]),
        start=1
    ):
        print(f"Rank: {rank}")
        print(f"Cosine distance: {distance:.4f}")
        print(document)
        print("-" * 60)